# Data harmonisation

The `unmapped entity` will be converted into `linked_eneties` with the link with other datasets (i.e. secondary, supplementary, and linked_entity).

The harmonisation process:
1. read the unmapped entities, one by one
2. link the unmapped entity one by one, by linking each key property; when link to the dataset, check the existing data
    - if there are existing data (e.g. technology), then use the existing one
    - if no existing data, then create a few on

In [1]:
import os
import pandas as pd
import yaml

# Load data

## data 1: all schema

In [2]:
# Define the base directory for schema files
base_dir = '../schema/'

# Initialize an empty dictionary to hold all loaded schemas
all_schemas = {}

# Function to recursively load YAML files from a given directory
def load_yaml_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.yaml'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    schema = yaml.safe_load(f)
                    all_schemas[file] = schema

# Call the function to load all YAML files from the schema directory and its subdirectories
load_yaml_files(base_dir)

# Print the number of loaded schemas for verification
print(f"Loaded {len(all_schemas)} schema files.")

Loaded 13 schema files.


## data 2: unmapped entity

In [3]:
## load the unmapped entities from the yaml file
path = r'../motel-db\unmapped_entity\unmapped_entities_refuel.yaml'

with open(path, "r", encoding="utf-8") as f:
    ue = yaml.safe_load(f)

print(f"Unmapped entities: {len(ue)}")

Unmapped entities: 99


## data 3: existing datasets

In [4]:
# Define the base directory for data files
data_dir = '../motel-db/'

# Initialize an empty dictionary to hold all loaded CSV data
all_csv_data = {}

# Function to recursively load CSV files from a given directory
def load_csv_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                with open(file_path, mode='r', encoding='utf-8') as f:
                    try:
                        data = pd.read_csv(f)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
                        data = None
                        continue
                    # Use the filename (without path/extension) or full path as key
                    file_name = file.split('.')[0]  # or use file_path for full paSth
                    all_csv_data[file_name] = data

# Load all CSV files from the motel-db directory and its subdirectories
load_csv_files(data_dir)

# Print the number of loaded CSV files for verification
print(f"Loaded {len(all_csv_data)} CSV files.")

Loaded 11 CSV files.


In [5]:
all_csv_data

{'attribute': Empty DataFrame
 Columns: [attribute_id, attribute_name, attribute_description, unit, data_format, ontology_iri, note]
 Index: [],
 'capacity_scope': Empty DataFrame
 Columns: [capacity_scope, capacity_scope_description, note]
 Index: [],
 'carrier': Empty DataFrame
 Columns: [carrier_id, carrier_name, carrier_description, carrier_type, carrier_category, carrier_reference, note]
 Index: [],
 'geographic_scope': Empty DataFrame
 Columns: [geographic_scope, geographic_scope_description, note]
 Index: [],
 'system_boundary': Empty DataFrame
 Columns: [system_boundary, system_boundary_description, note]
 Index: [],
 'temporal_scope': Empty DataFrame
 Columns: [temporal_scope, temporal_scope_description, note]
 Index: [],
 'process': Empty DataFrame
 Columns: [process_id, process_name, process_description, process_type, process_category, main_sector, feedstocks, products]
 Index: [],
 'source': Empty DataFrame
 Columns: [source_id, source_name, source_description, source_type,

# Harmonisation process

In [6]:
import ollama

In [7]:
ue[0]

{'technology_name': 'NH3_CCGT_El',
 'technology': {'technology_description': None,
  'technology_type': 'Conversion',
  'technology_category': 'Gas turbine',
  'technology_assumption': None,
  'process_name': 'CCGT',
  'process_type': None,
  'process_category': None,
  'process_assumption': None},
 'scope': {'geographic_scope_description': 'ECA',
  'temporal_scope_description': '2050',
  'capacity_scope_description': None,
  'system_boundary_description': 'Plant ready to operate',
  'scope_assumption': 'Mature'},
 'sources': [{'source_name': 'report_power_to_ammonia_2018',
   'source_description': 'Power-to-ammonia in future North European 100 %  renewable power and heat system',
   'linked_attribute': ['capex_per_capacity',
    'lifetime_yr',
    'opex_per_capacity_yr',
    'opex_per_energy',
    'technical_efficiency']},
  {'source_name': 'conversion_parametrization',
   'source_description': 'Calculating Input Output Ratios of Reactions',
   'linked_attribute': ['ratios_in', 'ratio

In [ ]:
import csv
import json
import os
import ollama

# --- 1. SETUP CONFIGURATION & TARGET VARIABLE ---
MODEL_NAME = "qwen3:14b"
REGISTRY_PATH = "secondary/technology.csv"

ue = {
    'technology_name': 'NH3_CCGT_El',
    'technology': {
        'technology_description': None,
        'technology_type': 'Conversion',
        'technology_category': 'Gas turbine',
        'technology_assumption': None,
        'process_name': 'CCGT',
        'process_type': None,
        'process_category': None,
        'process_assumption': None
    }
}

# --- 2. DATABASE ACCESS HELPERS ---
def load_all_existing_technologies():
    """Reads all current rows from the CSV registry to pass to the LLM context."""
    if not os.path.exists(REGISTRY_PATH):
        return []
    with open(REGISTRY_PATH, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)

def generate_next_tech_id():
    """Generates a sequential primary key if the LLM decides it is a new entry."""
    if not os.path.exists(REGISTRY_PATH):
        return "TECH_00001"
    with open(REGISTRY_PATH, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    return f"TECH_{len(rows) + 1:05d}"

def append_to_registry(row_dict):
    """Writes a newly approved row to the CSV registry file."""
    os.makedirs(os.path.dirname(REGISTRY_PATH), exist_ok=True)
    file_exists = os.path.exists(REGISTRY_PATH)
    
    with open(REGISTRY_PATH, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["tech_id", "technology_name", "technology_type", "technology_category", "technology_description"])
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# --- 3. LLM-BASED MATCHING & HARMONISATION ORCHESTRATOR ---
def harmonise_technology_with_llm(unmapped_entity):
    tech_name = unmapped_entity.get('technology_name')
    raw_tech_details = unmapped_entity.get('technology', {})
    
    # Load what we currently have in the database
    existing_database_rows = load_all_existing_technologies()
    
    print(f"🤖 Routing to Ollama ({MODEL_NAME}) to perform semantic matching and normalisation...")
    
    system_instruction = """
    You are an advanced data linkage and entity resolution node for reFuel.ch. 
    Your primary job is to check if an incoming 'unmapped technology' matches an 'existing technology' in our database registry using semantic fuzzy matching.

    CRITICAL RULES:
    1. Respond ONLY with a valid JSON object. No explanations, no conversation, no markdown formatting outside of JSON codeblocks.
    2. If the incoming technology semantically matches an existing row (even with slight differences in casing, underscores, or abbreviations), set "match_found" to true and return its "tech_id".
    3. If there is no safe semantic match, set "match_found" to false, leave "tech_id" as null, and provide structured, cleaned properties for creating a new database row.
    4. Strip out any attributes containing null, blank, or None values from the new properties.
    """
    
    user_prompt = f"""
    --- CURRENT DATABASE REGISTRY ROWS ---
    {json.dumps(existing_database_rows, indent=2)}

    --- INCOMING UNMAPPED TECHNOLOGY ---
    Name: "{tech_name}"
    Details: {json.dumps(raw_tech_details)}

    OUTPUT EXPECTED ENVELOPE FORMAT:
    {{
      "match_found": true,
      "tech_id": "TECH_00001",
      "cleaned_properties": null
    }}
    OR (If no match is found):
    {{
      "match_found": false,
      "tech_id": null,
      "cleaned_properties": {{
        "technology_type": "...",
        "technology_category": "...",
        "technology_description": "..."
      }}
    }}
    """
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_prompt}
            ],
            options={"temperature": 0.0}  # Maintain deterministic execution boundaries
        )
        
        # Parse and sanitize output
        raw_content = response['message']['content'].strip()
        if "```json" in raw_content:
            raw_content = raw_content.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_content:
            raw_content = raw_content.split("```")[1].split("```")[0].strip()
            
        decision = json.loads(raw_content)
        
        # --- BRANCH BASED ON LLM RESOLUTION ---
        if decision.get("match_found") is True:
            resolved_id = decision.get("tech_id")
            print(f"✅ LLM Match Found! Linked back to existing database primary key: {resolved_id}\n")
            return resolved_id
        
        else:
            print("❌ LLM determined this is a brand-new technology configuration entry. Creating row...")
            new_tech_id = generate_next_tech_id()
            props = decision.get("cleaned_properties", {})
            
            new_registry_row = {
                "tech_id": new_tech_id,
                "technology_name": tech_name,
                "technology_type": props.get("technology_type", "Unknown"),
                "technology_category": props.get("technology_category", "Unknown"),
                "technology_description": props.get("technology_description", "Normalised via Ollama pipeline loop")
            }
            
            append_to_registry(new_registry_row)
            print(f"🎉 Created new master row entry inside '{REGISTRY_PATH}':")
            print(json.dumps(new_registry_row, indent=2))
            return new_tech_id
            
    except Exception as e:
        print(f"💥 Matching Layer Failure: {e}")
        return None

🤖 Routing to Ollama (qwen3:14b) to perform semantic matching and normalisation...
❌ LLM determined this is a brand-new technology configuration entry. Creating row...
🎉 Created new master row entry inside 'secondary/technology.csv':
{
  "tech_id": "TECH_00001",
  "technology_name": "NH3_CCGT_El",
  "technology_type": "Conversion",
  "technology_category": "Gas turbine",
  "technology_description": "Normalised via Ollama pipeline loop"
}
🔗 Verification Complete. Active linked token variable tech_id: TECH_00001


In [10]:
# --- 4. EXECUTION ---
if __name__ == "__main__":
    resolved_id = harmonise_technology_with_llm(ue)
    print(f"🔗 Verification Complete. Active linked token variable tech_id: {resolved_id}")

🤖 Routing to Ollama (qwen3:14b) to perform semantic matching and normalisation...
✅ LLM Match Found! Linked back to existing database primary key: TECH_00001

🔗 Verification Complete. Active linked token variable tech_id: TECH_00001
